# Project Report: Analysis of Assets and Re-Contestation of Members of Parliament (2014–2024) 

## A Comparative Study of Andhra Pradesh, Bihar, and Gujarat across Three General Elections 
### Data Extraction, OCR Processing, Cleaning, Imputation, Deduplication, Candidate Matching, and Visualization Dashboard 

## 1. Introduction & Overview 
* In a democratic system like India, transparency regarding elected representatives' personal backgrounds, assets, liabilities, and criminal antecedents is vital for public awareness and democratic accountability .
* Every candidate contesting the Lok Sabha elections files an official self-sworn affidavit with the Election Commission of India (ECI), detailing their financial profile, educational qualifications, age, and criminal cases .
* Web platforms such as MyNeta (managed by the Association for Democratic Reforms) aggregate these affidavits for public scrutiny, while official election records and Wikipedia provide verified listings of winning candidates per constituency .
* This project presents a comprehensive computational pipeline built to track, clean, integrate, and analyze the financial asset trajectories of Parliamentary candidates (both winners and losers) across three major Indian states—Andhra Pradesh, Bihar, and Gujarat—over three consecutive Lok Sabha General Elections: 2014, 2019, and 2024 .
* The pipeline addresses significant real-world data engineering challenges, including web scraping raw text and HTML tables, resolving scanned image-based asset disclosures via Optical Character Recognition (OCR), performing statistical missing-data imputation, fuzzy candidate name deduplication, and longitudinal candidate matching across election cycles .
* Finally, an interactive visualization dashboard was created to present the findings .

---

## 2. Data Scraping Pipeline & Sources 
* The dataset encompasses candidate records across 9 distinct state-year combinations (3 states x 3 election years) .
* To construct a comprehensive database for comparison, data was collected from two primary sources :
  * **MyNeta Website Scraping**: Complete candidate lists, party affiliations, self-declared age, criminal cases, educational qualification, total movable and immovable assets, and liabilities were collected from the MyNeta platform .
  * **Wikipedia Election Results**: Official list of victorious Members of Parliament (MPs) across all Parliamentary Constituencies in Andhra Pradesh, Bihar, and Gujarat for 2014, 2019, and 2024 were scraped from Wikipedia and verified against ECI election reports .
  * **Optical Character Recognition (EasyOCR)**: A significant challenge arose because several candidate affidavits on MyNeta displayed asset or liability totals as image photos (~ Crore+, ~ Lacs+) rather than raw numerical text . The python module easyOCR is used to scan the image files, extract numerical values, and populate raw CSV files . The OCR is done in collab to utilise T4 GPUs as easyOCR is built to work primarily on GPUs .
* Wikipedia allows scraping by respecting crawl delay. So we used time delay of 1 second. There is no terms of use for Wikipedia
* Myneta robots.txt doesnot prevent anyone from scraping, but its terms of use does. Since we are not using this data for commercial purpose and also never agreed on the terms of use, so we scraped using a 2 second time delay
* In HTML response of Myneta, so many rows were missing because they were dynamically rendered by Javascript, so we had to use selenium even though it gives a HTML response
---

## 3. Directory Strucutre

* The scraped files from **MyNeta Website** and **Wikipedia** are stored as raw data files in `raw` directory.
* The directory `interim` contains `ocr data`, `cleaned raw data` and `cleaned data`, the ocr data folder contains the files obtained after running the ocr notebook to extract assets and liability values from images.
* `cleaned raw data` contains the files after modifying columns for dtypes, cleaning candidate names and masking missing values as -1.
* `cleaned data` folder contains the files after dropping the missing rows.
* `clean` directory contains the final data files used for dashboard after matching for recontester across election.

## 4. Data Cleaning & Normalization 
* Raw scraped electoral datasets contained substantial noise, inconsistent formatting, honorific prefixes, and non-numeric representations of financial figures .
* Structured Python helper functions were engineered using standard libraries (`pandas`, `re`) to standardize all columns .

### 4.1 Candidate Name Normalization (`clean_candidate`) 
* Candidate names in Indian political affidavits often include formal titles (e.g., 'Adv.', 'Dr.', 'Shri', 'Smt.', 'Prof.') as well as lineage/relationship indicators ('S/o', 'D/o', 'W/o', 'C/o', 'H/o') or alias markers ('@') .
* The `clean_candidate` helper function uses regular expressions to strip prefixes, remove lineage details and parentheses, replace dots with spaces, and title-case the remaining words to create a clean, uniform string representation .

### 4.2 Numeric Conversion of Age and Financial Columns (`clean_age` & `clean_money`) 
* **Age Cleaning**: Age strings were coerced into standard numeric integers using `pd.to_numeric` . Unparseable or missing values were assigned a placeholder sentinel of -1 .
* **Asset & Liability Cleaning**: Financial strings in MyNeta raw data contain formatting artifacts such as `Rs 1,76,20,000\r\n~ 1 Crore+` . The `clean_money` function extracted numeric digits and commas following the 'Rs' symbol using regular expression matching, removed formatting commas, and converted the extracted values into plain integers representing total Rupees . Unparseable fields were flagged with -1 .

---

## 5. Candidate Deduplication Strategy 
* Duplicate candidate entries within a single election dataset can distort asset distributions and re-contestation matching .
* To detect subtle duplicate filings (e.g., slightly varied spelling of names or minor age discrepancies across multiple filings), a custom fuzzy matching deduplication algorithm (`find_dups`) was implemented using the RapidFuzz library .
* **Name Normalization (`std_name`)**: Standardizes candidate names, party names, and education levels to lowercase and removes punctuation .
* **Name Agreement Logic (`names_agree`)**: Evaluates candidate name similarity using word permutation and Jaro-Winkler distance with an 85% similarity threshold (0.85) . It allows full word matches even when name word orders differ .
* **Pairwise Matching Rule**: Pairs of rows are compared within a single state-year dataset . A candidate pair is flagged as a duplicate if candidate age difference $\le 1$ year, criminal case counts match exactly, political party affiliation matches, and names satisfy the fuzzy Jaro-Winkler criteria .
* Through manual inspection of flagged pairs, confirmed duplicates were removed across the datasets (e.g., dropping duplicate index 65 in Andhra 2014, index 87 in Andhra 2019, index 430 in Andhra 2024, indices 591 & 588 in Bihar 2019, index 246 in Gujarat 2014, and index 173 in Gujarat 2024) .

---

## 6. Missing Value Tests
* Before performing statistical analysis, missing data patterns across Age, Total Assets, and Liabilities were evaluated using Seaborn heatmap visualizations .
* Missing values (represented by NaN or sentinel -1) were identified across several state-year datasets .
* The missingness was found to be MAR. As there are very less missing values and all the missing rows belong only to losing candidates we decided to drop the rows .

---

## 7. Candidate & Electoral Record Matching 
* A central component of the analytical workflow involved merging asset disclosures with election outcomes and linking candidates who re-contested across multiple election cycles (2014 to 2019, 2019 to 2024, or 2014 to 2024) .
* A standardized lookup/matching file (`tomatch`) was developed to establish explicit linkages :
  * **Winner/Loser Classification**: Scraped winner names from Wikipedia were matched against candidate names in the MyNeta asset datasets for each constituency . This labeled each candidate as either 'Winner' or 'Loser' .
  * **Re-contestation Tracking**: Candidates contesting in consecutive elections were tracked by matching normalized names, party history, constituency, and expected age progression (+5 years between cycles) . This enabled tracking individual asset growth for re-contesting MPs .
  * **Asset Dynamics Analysis**: Calculated absolute asset change ($\text{Assets}_{t2} - \text{Assets}_{t1}$) and percentage growth rate ($(\text{Assets}_{t2} - \text{Assets}_{t1}) / \text{Assets}_{t1} \times 100$) for re-contesting candidates .

---

## 8. Visualization Dashboard & Analytical Insights (old)
* To make the processed data accessible and actionable, an interactive data visualization dashboard was created .
* The dashboard visualizes key financial trends, comparisons, and socio-demographic metrics across states and election years .
* **Key Dashboard Module 1**: Winner vs. Loser Asset Comparison: Bar charts and box plots contrasting the median and mean net assets of winning candidates against losing candidates across Andhra Pradesh, Bihar, and Gujarat .
* **Key Dashboard Module 2**: Re-contesting Candidates Asset Growth: Scatter plots and growth distributions highlighting asset inflation for MPs re-contesting in 2019 and 2024 .
* **Key Dashboard Module 3**: Party-Wise Financial Disparities: Distribution of wealth categorized by political parties across the three states .
* **Key Dashboard Module 4**: Criminal Cases & Financial Correlation: Analysis comparing candidate asset levels with declared criminal charges .

## 9. Visualization Dashboard & Analytical Insights
* To make the processed data accessible and actionable, an interactive dark-themed Dash web application (`dash_with_statemap.py`) was created .
* The dashboard visualizes key financial trends, comparisons, and socio-demographic metrics across states and election years through multi-view navigation tabs .
* **Key Dashboard Module 1: State-Level Returning Candidate Comparisons (`/andhra-pradesh`, `/bihar`, `/gujarat`)**: Features dual-period filtering (2014–2019, 2019–2024, or both intervals combined) tracking percentage and absolute asset changes for candidates who re-contested, paired with mean asset change bar charts and equal-width binned histograms comparing winners versus losers, A Horizontal Bar graph of constituencies corresponding to Asset change of its winners .
* **Key Dashboard Module 2: Constituency Map & Geographic Visualization (`/constituency-map`)**: Loads simplified GeoJSON parliamentary boundaries (`india_pc_2019_simplified.geojson`) based on the 2008 delimitation . It visualizes declared asset growth or decline for sitting MPs across constituencies using a compressed inverse hyperbolic sine (`asinh`) diverging colour scale, supported by interactive data-quality tiles tracking linked versus unlinked seats .
* **Key Dashboard Module 3: Party-Wise Financial Disparities (`/parties`)**: Aggregates candidate counts and total asset wealth by election result across states . It utilizes log-scale grouped bar charts to contrast median candidate assets between winners and non-winners alongside interactive donut charts (with hole radius $0.55$) displaying top party wealth shares while grouping independents (`IND`) and smaller groups into an "Others" category .
* **Key Dashboard Module 4: Data-Cleaning Pipeline Audit (`/cleaning`)**: Provides complete transparency into data engineering steps by tracking raw scraped row counts versus successfully parsed numerical entries and automated EasyOCR recovery files, visualizing positive asset distributions across pipeline stages via logarithmic box plots .

### Key Findings 
* **Asset Disparity**: Winning MPs across all three states exhibited **significantly higher** median assets compared to losing candidates in all three election cycles (2014, 2019, and 2024) .
* **Re-contestation Growth**: Re-contesting candidates demonstrated positive asset growth across election terms, with re-contesting winners exhibiting a **significantly higher** median asset increase compared to non-winning re-contesting candidates .
* **State-Level Variations**: Andhra Pradesh recorded the highest average candidate asset disclosures among the three states, whereas Bihar exhibited lower overall median asset values alongside higher variability in declared criminal cases .

## Contributions
* Nivas DA25B007: Myneta scraping , Visualisation, Data cleaning, recontest matching
* Gagan DA25B012: Wikipedia scraping, Missingness Tests, Dashboard Design
* Balaram DA25B011: Assets OCR, recontest matcing, deduplication, final report

* 2014 data: 17th oct
* 2019 data: 17th oct
* 2024 data: 20th oct
* winners of 2019, 2014: 18th oct
* winners of 2024: 25th oct

# DASHBOARD

In [2]:
from dashboard_bivariate import app
app.run(jupyter_mode='inline')